In [7]:
class VideoCaptionDataset(Dataset):
    def __init__(self, csv_file, npz_dir, tokenizer):
        self.df = pd.read_csv(csv_file)
        self.npz_dir = npz_dir
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_id = row["video_id"]
        caption = row["caption"]

        npz_path = os.path.join(self.npz_dir, f"{video_id}.npz")
        data = np.load(npz_path)
        
        # Expected shape from npz: (8, 3, 224, 224)
        frames = data["video"].astype(np.float32) / 255.0
        
        # Convert to tensor (Frames, C, H, W)
        frames = torch.tensor(frames)

        tokens = self.tokenizer(
            caption,
            padding="max_length",
            max_length=20,
            truncation=True,
            return_tensors="pt"
        )

        labels = tokens.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": frames, # (8, 3, 224, 224)
            "labels": labels
        }

In [16]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Configuration
INPUT_DIR = "/Users/ayraj/Desktop/video_captioning/processed_frames"
OUTPUT_DIR = "/Users/ayraj/Desktop/video_captioning/final_frames_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Refactoring frames for VideoMAE compatibility...")

for file_name in tqdm(os.listdir(INPUT_DIR)):
    if file_name.endswith(".npz"):
        file_path = os.path.join(INPUT_DIR, file_name)
        
        # 1. Load existing data
        data = np.load(file_path)
        video = data['video'] # Expected: (8, 3, H, W) or (8, H, W, 3)
        
        # 2. Convert to Tensor for easy manipulation
        video_tensor = torch.from_numpy(video).float()
        
        # 3. Ensure Channels are at the correct index (C, T, H, W)
        # If your data is (8, 224, 224, 3), permute(3, 0, 1, 2)
        # If your data is (8, 3, 224, 224), permute(1, 0, 2, 3)
        if video_tensor.shape[1] == 3:
            video_tensor = video_tensor.permute(1, 0, 2, 3)
        elif video_tensor.shape[-1] == 3:
            video_tensor = video_tensor.permute(3, 0, 1, 2)
            
        # 4. Enforce 224x224 resolution
        if video_tensor.shape[-2:] != (224, 224):
            # Interpolate spatial dimensions only
            video_tensor = F.interpolate(video_tensor, size=(224, 224), mode='bilinear')
            
        # 5. Save back to NPZ
        # Shape is now strictly (3, 8, 224, 224)
        output_path = os.path.join(OUTPUT_DIR, file_name)
        np.savez_compressed(output_path, video=video_tensor.numpy().astype(np.uint8))

print(f"Refactoring complete. New files saved in: {OUTPUT_DIR}")

Refactoring frames for VideoMAE compatibility...


100%|██████████| 2000/2000 [01:02<00:00, 32.19it/s]

Refactoring complete. New files saved in: /Users/ayraj/Desktop/video_captioning/final_frames_v2


In [17]:
import numpy as np
import os

# Path to your new frames
CHECK_DIR = "/Users/ayraj/Desktop/video_captioning/final_frames_v2"
sample_file = os.path.join(CHECK_DIR, os.listdir(CHECK_DIR)[0])

data = np.load(sample_file)
frames = data['video']

print(f"File: {sample_file}")
print(f"Shape: {frames.shape}")  # MUST BE (3, 8, 224, 224)
print(f"Data Type: {frames.dtype}")
print(f"Value Range: {frames.min()} to {frames.max()}") # Should be 0-255

if frames.shape == (3, 8, 224, 224):
    print("✅ SUCCESS: Format is usable for VideoMAE.")
else:
    print("❌ ERROR: Shape mismatch. Model will throw a ValueError.")

File: /Users/ayraj/Desktop/video_captioning/final_frames_v2/video2963.npz
Shape: (3, 8, 224, 224)
Data Type: uint8
Value Range: 0 to 255
✅ SUCCESS: Format is usable for VideoMAE.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
CSV_PATH = "/Users/ayraj/Desktop/video_captioning/msrvtt_2k_preprocessed.csv"
NPZ_DIR = "/Users/ayraj/Desktop/video_captioning/final_frames_v2"
OUTPUT_DIR = "blip_video_model_2"

print("Loading BLIP Processor and Model...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


# Dataset (Random Frame Sampling)

class BLIP_Video_Dataset(Dataset):
    def __init__(self, csv_file, npz_dir, processor):
        self.df = pd.read_csv(csv_file)
        self.npz_dir = npz_dir
        self.processor = processor

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        npz_path = os.path.join(self.npz_dir, f"{row['video_id']}.npz")
        
        with np.load(npz_path) as data:
            video = data['video'] # Shape: (3, 8, 224, 224)
            
        # THE TRICK: Pick 1 random frame from the 8 available
        frame_idx = np.random.randint(0, video.shape[1])
        frame = video[:, frame_idx, :, :] # New Shape: (3, 224, 224)
        
        # Convert to (H, W, C) for the BLIP Processor
        frame = np.transpose(frame, (1, 2, 0))
        
        # Ensure it is properly scaled for the processor (0-255 uint8)
        if frame.max() <= 1.0:
            frame = (frame * 255).astype(np.uint8)
        else:
            frame = frame.astype(np.uint8)
            
        # Let the processor handle the image formatting and text tokenization
        encoding = self.processor(
            images=frame, 
            text=row['caption'], 
            padding="max_length", 
            max_length=30, 
            return_tensors="pt", 
            truncation=True
        )
        
        # Extract the input_ids
        input_ids = encoding.input_ids.squeeze()
        
        # BLIP needs the text tokens passed as 'labels' for training, with padding set to -100
        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {
            "pixel_values": encoding.pixel_values.squeeze(),
            "input_ids": input_ids, # <-- The fix is right here!
            "labels": labels
        }


#  Memory Optimization

# BLIP's vision encoder is already perfect at seeing images. 
# We freeze it so your Mac only spends power training the text side.
for param in model.vision_model.parameters():
    param.requires_grad = False

print("Vision encoder frozen. Fully training the text decoder!")
model.to(DEVICE)


# Training Loop

train_loader = DataLoader(BLIP_Video_Dataset(CSV_PATH, NPZ_DIR, processor), 
                          batch_size=4, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

print(f"Starting BLIP Training on {DEVICE}...")
model.train()

for epoch in range(3):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in loop:
        pixel_values = batch["pixel_values"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE) # <-- Catching the input_ids
        labels = batch["labels"].to(DEVICE)

        # Pass all three to the model
        outputs = model(pixel_values=pixel_values, input_ids=input_ids, labels=labels)
        
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} Completed! Average Loss: {avg_loss:.4f}")
    
    
    # SAVE CHECKPOINT AFTER EACH EPOCH
  
    epoch_dir = f"{OUTPUT_DIR}_epoch_{epoch+1}"
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    print(f"Checkpoint successfully saved to: {epoch_dir}\n")
    
    # Clear Apple Silicon memory cache
    if DEVICE == "mps":
        torch.mps.empty_cache()


# Save Final

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Training Complete! Final model saved to: {OUTPUT_DIR}")

Loading BLIP Processor and Model...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 74935.06it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when 

Vision encoder frozen. Fully training the text decoder!
Starting BLIP Training on mps...


Epoch 1: 100%|██████████| 10000/10000 [53:43<00:00,  3.10it/s, loss=3.05] 



Epoch 1 Completed! Average Loss: 3.0709


...


Epoch 3 Completed! Average Loss: 2.0260


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.42it/s]


Checkpoint successfully saved to: blip_video_model_2_epoch_3



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]

Training Complete! Final model saved to: blip_video_model_2
